# vLLM Serving & AWQ Quantisation on Colab T4

This notebook sets up the model, quantises with AWQ, and runs a benchmark.

**Runtime**: Runtime → Change runtime type → T4 GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
NOTEBOOK_DIR = "/content/frontier-llmops-core"
DRIVE_OUTPUT = "/content/drive/MyDrive/serving-output"

if not os.path.exists(NOTEBOOK_DIR):
    !git clone https://github.com/anmolsharma152/frontier-llmops-core.git {NOTEBOOK_DIR}
%cd {NOTEBOOK_DIR}

In [ ]:
!pip install -e 05_serving_inference/
!pip install autoawq -q

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")

### AWQ Quantisation

In [ ]:
!mkdir -p {DRIVE_OUTPUT}
!python 05_serving_inference/scripts/main.py --config 05_serving_inference/configs/defaults.yaml quantize-awq --output-path {DRIVE_OUTPUT}/awq-model
print(f"Quantised model saved to {DRIVE_OUTPUT}/awq-model")

### Start vLLM Server (background)

In [ ]:
import subprocess, time
server_proc = subprocess.Popen(
    ["python", "05_serving_inference/scripts/main.py", "--config", "05_serving_inference/configs/defaults.yaml", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(30)  # wait for model to load
print("Server started")

In [ ]:
!python 05_serving_inference/scripts/main.py --config 05_serving_inference/configs/defaults.yaml client --prompt "What is deep learning?"

In [ ]:
server_proc.terminate()
print("Server stopped")